# Stage 06 - Data Preprocessing

**Dataset:** `data/raw/sample_data.csv` (7 rows, 6 columns)
**Cleaning code:** `src/cleaning.py` - `fill_missing_median`, `drop_missing`, `normalize_data`

The order of operations below is deliberate, and the order changes the answer:

1. **Repair encodings first** - turn disguised missing values (`'Unknown'`) into real `NaN`
   and merge duplicate labels (`'SF'` / `'San Francisco'`). Do this *before* counting
   missingness, or the counts are wrong.
2. **Drop what is too sparse to trust** - columns first, then rows. Dropping a
   mostly-empty column shrinks the denominator that rows are judged against.
3. **Fill what is left** - median imputation on the survivors only. Filling before
   dropping would compute medians from rows that are about to be deleted.
4. **Scale last** - and into a separate file, because scaling is a modeling choice,
   not a statement about what the data *is*.

In [1]:
# --- setup ---
import os
import sys

import numpy as np
import pandas as pd

pd.set_option('display.width', 120)
np.random.seed(6)   # nothing here is random today, but the seed keeps re-runs identical

# make sure `from src import cleaning` works no matter how the kernel was launched
if '' not in sys.path:
    sys.path.insert(0, '')

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

print('working from:', os.path.basename(os.getcwd()))

working from: homework06


In [2]:
# --- generate the raw dataset if it is not already there (from the starter) ---
raw_path = 'data/raw/sample_data.csv'

if not os.path.exists(raw_path):
    seed_data = {
        'age':        [34, 45, 29, 50, 38, np.nan, 41],
        'income':     [55000, np.nan, 42000, 58000, np.nan, np.nan, 49000],
        'score':      [0.82, 0.91, np.nan, 0.76, 0.88, 0.65, 0.79],
        'zipcode':    ['90210', '10001', '60614', '94103', '73301', '12345', '94105'],
        'city':       ['Beverly', 'New York', 'Chicago', 'SF', 'Austin', 'Unknown',
                       'San Francisco'],
        'extra_data': [np.nan, 42, np.nan, np.nan, np.nan, 5, np.nan],
    }
    pd.DataFrame(seed_data).to_csv(raw_path, index=False)
    print('created', raw_path)
else:
    print('already present:', raw_path)

already present: data/raw/sample_data.csv


In [3]:
from src import cleaning

print(cleaning.fill_missing_median.__doc__.splitlines()[0])
print(cleaning.drop_missing.__doc__.splitlines()[0])
print(cleaning.normalize_data.__doc__.splitlines()[0])

Fill missing values in numeric columns with that column's median.
Drop rows or columns based on how much data they are missing.
Put numeric columns on a common scale.


## Step 1 - Load and inspect

`zipcode` is read as a **string on purpose**. Left to itself pandas reads `90210` as an
integer, which is wrong in a way that does not announce itself: zip codes are labels, not
quantities. Averaging them is meaningless, and `01001` silently becomes `1001`.

In [4]:
raw = pd.read_csv('data/raw/sample_data.csv', dtype={'zipcode': 'string'})

print('shape:', raw.shape)
raw.info()
raw

shape: (7, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   age         6 non-null      float64
 1   income      4 non-null      float64
 2   score       6 non-null      float64
 3   zipcode     7 non-null      string 
 4   city        7 non-null      object 
 5   extra_data  2 non-null      float64
dtypes: float64(4), object(1), string(1)
memory usage: 464.0+ bytes


,age,income,score,zipcode,city,extra_data
0,34.0,55000.0,0.82,90210,Beverly,NaN
1,45.0,NaN,0.91,10001,New York,42.0
2,29.0,42000.0,NaN,60614,Chicago,NaN
3,50.0,58000.0,0.76,94103,SF,NaN
4,38.0,NaN,0.88,73301,Austin,NaN
5,NaN,NaN,0.65,12345,Unknown,5.0
6,41.0,49000.0,0.79,94105,San Francisco,NaN


In [5]:
# --- missingness report: the thing that drives every decision below ---
def missing_report(df):
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing': df.isna().sum(),
        'missing_pct': (df.isna().mean() * 100).round(1),
        'present_pct': ((1 - df.isna().mean()) * 100).round(1),
    })

missing_report(raw)

,dtype,missing,missing_pct,present_pct
age,float64,1,14.3,85.7
income,float64,3,42.9,57.1
score,float64,1,14.3,85.7
zipcode,string,0,0.0,100.0
city,object,0,0.0,100.0
extra_data,float64,5,71.4,28.6


### Assumptions recorded before touching anything

| # | Observation | Assumption I am making | Consequence if I am wrong |
|---|---|---|---|
| 1 | `extra_data` is 71.4% missing (2 of 7 values) | Two observations cannot support a column. It carries no usable signal. | If those two values were the whole point of the dataset, I have deleted the answer. I would need a data dictionary to rule that out. |
| 2 | `income` is 42.9% missing | Missing at random, so a median stand-in is honest. | If income is missing *because* it is high or low, median filling erases exactly the pattern worth studying. |
| 3 | `city` contains the literal string `'Unknown'` | It is a disguised missing value, not a place. | If `'Unknown'` is a real category the source uses deliberately, I have converted meaning into absence. |
| 4 | `city` contains both `'SF'` and `'San Francisco'` | Same city, two spellings. | Low risk here; the two would otherwise be counted as separate cities. |
| 5 | `zipcode` looks numeric | It is an identifier, so it stays text. | Reading it as an integer drops leading zeros and invites nonsense arithmetic. |

The threshold used for "too sparse" is **50% present**, applied to columns and then to rows.
That number is a judgement, not a law - it is stated here so a reader can disagree with it
in one place rather than reverse-engineer it from the code.

## Step 2 - Repair encodings before measuring anything

`'Unknown'` is missingness wearing a costume. Until it is converted, `city` reports 0%
missing and every downstream count is off.

In [6]:
repaired = raw.copy()

repaired['city'] = repaired['city'].replace('Unknown', pd.NA)      # disguised missing -> real missing
repaired['city'] = repaired['city'].replace({'SF': 'San Francisco'})  # merge duplicate label

print('city before:', raw['city'].tolist())
print('city after :', repaired['city'].tolist())
print()
print('city missing before:', int(raw['city'].isna().sum()),
      '-> after:', int(repaired['city'].isna().sum()))
print('distinct cities before:', raw['city'].nunique(),
      '-> after:', repaired['city'].nunique())

city before: ['Beverly', 'New York', 'Chicago', 'SF', 'Austin', 'Unknown', 'San Francisco']
city after : ['Beverly', 'New York', 'Chicago', 'San Francisco', 'Austin', <NA>, 'San Francisco']

city missing before: 0 -> after: 1
distinct cities before: 7 -> after: 5


## Step 3 - Drop what is too sparse to trust

`drop_missing(threshold=0.5, axis='columns')` keeps columns that are at least half
populated; then `axis='rows'` does the same for rows. Columns go first on purpose: once
`extra_data` is gone, every row is judged against 5 columns instead of 6, so a row is no
longer punished for a blank in a column nobody is keeping.

In [7]:
present_by_col = (1 - repaired.isna().mean()).round(3)
print('fraction present, by column:')
print(present_by_col.to_string())

cols_kept = cleaning.drop_missing(repaired, threshold=0.5, axis='columns')

dropped_cols = [c for c in repaired.columns if c not in cols_kept.columns]
print('\ndropped columns:', dropped_cols)
print('kept columns   :', list(cols_kept.columns))

fraction present, by column:
age           0.857
income        0.571
score         0.857
zipcode       1.000
city          0.857
extra_data    0.286

dropped columns: ['extra_data']
kept columns   : ['age', 'income', 'score', 'zipcode', 'city']


In [8]:
present_by_row = (1 - cols_kept.isna().mean(axis=1)).round(3)
print('fraction present, by row:')
print(present_by_row.to_string())

rows_kept = cleaning.drop_missing(cols_kept, threshold=0.5, axis='rows')

dropped_rows = [i for i in cols_kept.index if i not in rows_kept.index]
print('\ndropped rows:', dropped_rows)
print('shape', cols_kept.shape, '->', rows_kept.shape)
print()
print('the dropped row(s):')
print(cols_kept.loc[dropped_rows])

fraction present, by row:
0    1.0
1    0.8
2    0.8
3    1.0
4    0.8
5    0.4
6    1.0

dropped rows: [5]
shape (7, 5) -> (6, 5)

the dropped row(s):
   age  income  score zipcode  city
5  NaN     NaN   0.65   12345  <NA>


Row 5 is the only casualty, and it deserved it: no `age`, no `income`, and a `city` that
was the string `'Unknown'`. Two of five fields present. Whatever that row is, it is not
evidence - imputing three of its five values would have been inventing a person.

## Step 4 - Fill what is left

`fill_missing_median` replaces the remaining numeric blanks with each column's median.
The median is used rather than the mean because on six rows a single extreme value moves
the mean a long way and the median barely at all.

The medians are computed **after** the drops, so they describe the data actually being
kept.

In [9]:
medians_used = rows_kept.select_dtypes(include='number').median()
print('medians used for filling:')
print(medians_used.to_string())

filled = cleaning.fill_missing_median(rows_kept)

print('\nmissing values before fill:', int(rows_kept.isna().sum().sum()))
print('missing values after fill :', int(filled.isna().sum().sum()))
print()
filled

medians used for filling:
age          39.50
income    52000.00
score         0.82

missing values before fill: 3
missing values after fill : 0



,age,income,score,zipcode,city
0,34.0,55000.0,0.82,90210,Beverly
1,45.0,52000.0,0.91,10001,New York
2,29.0,42000.0,0.82,60614,Chicago
3,50.0,58000.0,0.76,94103,San Francisco
4,38.0,52000.0,0.88,73301,Austin
6,41.0,49000.0,0.79,94105,San Francisco


In [10]:
# --- exactly which cells were invented, and what went into them ---
was_missing = rows_kept.isna()
changes = []
for col in rows_kept.columns:
    for idx in rows_kept.index[was_missing[col]]:
        changes.append({'row': idx, 'column': col,
                        'filled_with': filled.loc[idx, col]})

pd.DataFrame(changes)

,row,column,filled_with
0,1,income,52000.00
1,4,income,52000.00
2,2,score,0.82


## Step 5 - Normalize, into a separate artifact

Scaling is not a fact about the data, it is a preparation for a particular model. A tree
does not care about scale; a distance-based model or a regularised regression cares a lot.
So the scaled frame is produced here and saved separately - the canonical cleaned file
keeps dollars as dollars and years as years, because that is the file a human reads.

In [11]:
num_cols = ['age', 'income', 'score']

scaled_minmax = cleaning.normalize_data(filled, columns=num_cols, method='minmax')
scaled_standard = cleaning.normalize_data(filled, columns=num_cols, method='standard')

print('min-max: every column should span exactly 0 to 1')
print(scaled_minmax[num_cols].agg(['min', 'max']).round(3).to_string())
print()
print('standard: mean 0, and population std 1')
print(scaled_standard[num_cols].agg(['mean', lambda s: s.std(ddof=0)])
      .set_axis(['mean', 'std_ddof0']).round(6).to_string())

min-max: every column should span exactly 0 to 1
     age  income  score
min  0.0     0.0    0.0
max  1.0     1.0    1.0

standard: mean 0, and population std 1
           age  income  score
mean       0.0    -0.0    0.0
std_ddof0  1.0     1.0    1.0


A note so the numbers are not misread: `normalize_data(method='standard')` divides by the
**population** standard deviation (`ddof=0`), which is what scikit-learn's `StandardScaler`
does. Pandas' default `.std()` uses `ddof=1`, so checking the output with a plain `.std()`
returns about 1.095 on six rows, not 1.000. That is the sample-vs-population correction,
not a bug - hence the explicit `ddof=0` in the check above.

In [12]:
scaled_minmax.round(3)

,age,income,score,zipcode,city
0,0.238,0.812,0.4,90210,Beverly
1,0.762,0.625,1.0,10001,New York
2,0.000,0.000,0.4,60614,Chicago
3,1.000,1.000,0.0,94103,San Francisco
4,0.429,0.625,0.8,73301,Austin
6,0.571,0.438,0.2,94105,San Francisco


## Step 6 - Save

Two formats, for the reason established in Stage 05: **CSV is text and forgets dtypes.**
`zipcode` was deliberately loaded as a string, and a CSV round trip turns it straight back
into an integer. Parquet remembers. CSV is the required deliverable and the readable one;
Parquet is the one that survives.

In [13]:
filled.to_csv('data/processed/sample_data_cleaned.csv', index=False)
scaled_minmax.to_csv('data/processed/sample_data_scaled_minmax.csv', index=False)

parquet_ok = True
try:
    filled.to_parquet('data/processed/sample_data_cleaned.parquet', index=False)
except Exception as exc:
    parquet_ok = False
    print('parquet unavailable (needs pyarrow):', exc)

for name in sorted(os.listdir('data/processed')):
    size = os.path.getsize(os.path.join('data/processed', name))
    print(f'{name:<38} {size:>7,} bytes')

sample_data_cleaned.csv                    234 bytes
sample_data_cleaned.parquet              3,473 bytes
sample_data_scaled_minmax.csv              331 bytes


In [14]:
# --- the dtype lesson, demonstrated rather than asserted ---
print('zipcode dtype in memory        :', filled['zipcode'].dtype)

csv_back = pd.read_csv('data/processed/sample_data_cleaned.csv')
print('zipcode dtype after CSV round  :', csv_back['zipcode'].dtype)

if parquet_ok:
    pq_back = pd.read_parquet('data/processed/sample_data_cleaned.parquet')
    print('zipcode dtype after Parquet    :', pq_back['zipcode'].dtype)

print()
print('reload the CSV with an explicit schema and it is a string again:')
csv_typed = pd.read_csv('data/processed/sample_data_cleaned.csv',
                        dtype={'zipcode': 'string'})
print('zipcode dtype with dtype= arg  :', csv_typed['zipcode'].dtype)

zipcode dtype in memory        : string
zipcode dtype after CSV round  : int64
zipcode dtype after Parquet    : string

reload the CSV with an explicit schema and it is a string again:
zipcode dtype with dtype= arg  : string


## Step 7 - Original vs cleaned

In [15]:
comparison = pd.DataFrame({
    'original': [raw.shape[0], raw.shape[1], int(raw.isna().sum().sum()),
                 raw['city'].nunique(dropna=True), str(raw['zipcode'].dtype)],
    'cleaned':  [filled.shape[0], filled.shape[1], int(filled.isna().sum().sum()),
                 filled['city'].nunique(dropna=True), str(filled['zipcode'].dtype)],
}, index=['rows', 'columns', 'missing cells', 'distinct cities', 'zipcode dtype'])

comparison

,original,cleaned
rows,7,6
columns,6,5
missing cells,10,0
distinct cities,7,5
zipcode dtype,string,string


In [16]:
# --- did imputation move the distributions? ---
before = raw[num_cols].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']]
after = filled[num_cols].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']]

side_by_side = before.join(after, lsuffix='_before', rsuffix='_after')
side_by_side[['count_before', 'count_after',
              'mean_before', 'mean_after',
              'std_before', 'std_after',
              '50%_before', '50%_after']].round(2)

,count_before,count_after,mean_before,mean_after,std_before,std_after,50%_before,50%_after
age,6.0,6.0,39.5,39.50,7.56,7.56,39.5,39.50
income,4.0,6.0,51000.0,51333.33,7071.07,5501.51,52000.0,52000.00
score,6.0,6.0,0.8,0.83,0.09,0.06,0.8,0.82


## Reflection - what this cleaning cost

**Median filling shrinks variance, and the table above shows exactly where.** Only three
cells were actually invented: `income` in rows 1 and 4, and `score` in row 2. `income`'s
standard deviation falls from about 7,071 to 5,502 - a 22% drop - because two of six values
are now the same number sitting precisely at the centre. `age` is untouched (7.5565 before
and after) for the honest reason that its only blank was in the row that got dropped, so no
age was ever imputed. Downstream, any confidence interval built on `income` will be too
narrow: the imputed cells are pretending to be observations when they are one estimate
repeated. A model trained on this will look more certain than it has earned.

**Dropping is honest but expensive, and it moves numbers too.** Deleting row 5 cost 14% of
an already tiny sample. It also carried away the lowest `score` in the dataset (0.65), which
is most of why `score`'s standard deviation falls from 0.093 to 0.056 - that shrinkage is
the drop, not the imputation, and the two effects are easy to confuse when only the
before/after table is read. On 7 rows this matters; on 70,000 it would be noise. The
threshold that made sense here would be reckless on a dataset where every row is a customer.

**The 50% threshold is arbitrary and I would defend it only weakly.** It happens to
separate `extra_data` (28.6% present) from `income` (57.1% present) cleanly, which is
convenient - suspiciously so. With a threshold of 0.6 I would have dropped `income` too,
and the dataset would have lost its most interesting variable. The gap between those two
numbers is what makes 0.5 safe here, not the number 0.5 itself.

**What I would do with more data.** Add a missingness indicator column (`income_was_missing`)
before filling, so a model can learn whether absence itself predicts anything - that keeps
the information that median filling destroys. And I would test whether `income` is missing
at random by checking it against `city` and `score` rather than assuming it, which is what
assumption 2 above quietly does.

**What I deliberately did not do.** I did not scale the canonical cleaned file, and I did
not touch the `'Beverly'` value in `city` even though it is almost certainly meant to be
`'Beverly Hills'` for zip 90210. Correcting it would mean guessing at the source's intent
from one adjacent field, and a guess dressed as a repair is worse than a known oddity.